## Eviction & topology spread constraints

### Eviction

**Node-pressure eviction** — the kubelet watches signals (memory, disk, inodes) and, past thresholds, evicts to relieve pressure, in QoS order: `BestEffort` → `Burstable` (over requests) → `Guaranteed` last; within a tier, the biggest users of the pressed resource.

**API-initiated eviction** — `kubectl drain <node>` gracefully evicts every non-DaemonSet pod (before maintenance), calling the eviction API which respects PodDisruptionBudgets. `kubectl delete pod` is *not* an eviction and ignores PDBs.

**`PodDisruptionBudget`** caps how many pods may be *voluntarily* disrupted at once:

```yaml
kind: PodDisruptionBudget
spec:
  minAvailable: 2          # or maxUnavailable: 1
  selector: { matchLabels: { app: web } }
```

Drain refuses to evict if it would drop below `minAvailable`. PDBs only guard **voluntary** disruptions (drain, autoscaling) — not hardware failure.

### Topology spread constraints

A cleaner way than anti-affinity to say "spread evenly":

```yaml
topologySpreadConstraints:
  - maxSkew: 1                        # max imbalance between any two domains
    topologyKey: topology.kubernetes.io/zone
    whenUnsatisfiable: DoNotSchedule  # or ScheduleAnyway
    labelSelector: { matchLabels: { app: web } }
```

`maxSkew: 1` is the tightest. `DoNotSchedule` = hard (Pending if impossible); `ScheduleAnyway` = soft. Cluster defaults often spread across hostname + zone with `ScheduleAnyway` — the "free" zone spreading on managed clusters. Use **spread constraints** for "spread evenly," **anti-affinity** for "don't co-locate with a *different* workload." On our map both sit in the **Scheduling** box, shaping the scheduler's placement.